# 📥 Notebook 1: Data Collection & Preprocessing

**FinTech Stock Market Analysis Project**

This notebook covers:

- Downloading historical stock data for major FinTech companies
- Data cleaning and handling missing values
- Feature engineering (Moving Averages, RSI, MACD, Bollinger Bands)
- Saving processed data for downstream notebooks


In [21]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os
import warnings
warnings.filterwarnings('ignore')

# Create data directory
os.makedirs('../data', exist_ok=True)

print('✅ Libraries loaded successfully!')

✅ Libraries loaded successfully!


## 1. Define FinTech Stocks & Download Data


In [22]:
# ── Install this first in terminal if not done: pip install curl_cffi ──

import yfinance as yf
import pandas as pd
import numpy as np
import time

# Major FinTech companies
FINTECH_TICKERS = {
    'JPM':  'JPMorgan Chase',
    'V':    'Visa',
    'MA':   'Mastercard',
    'PYPL': 'PayPal',
    'XYZ':  'Block (Square)',
    'COIN': 'Coinbase',
    'HOOD': 'Robinhood',
    'AFRM': 'Affirm'
}

START_DATE = '2020-01-01'
END_DATE   = '2024-12-31'

print(f'Downloading data for {len(FINTECH_TICKERS)} FinTech stocks...')
print(f'Period: {START_DATE} → {END_DATE}\n')

raw_data = {}
for ticker, name in FINTECH_TICKERS.items():
    print(f'  Downloading {ticker} ({name})...')
    try:
        df = yf.download(
            ticker,
            start=START_DATE,
            end=END_DATE,
            progress=False,
            auto_adjust=True
        )
        # Flatten multi-level columns (newer yfinance versions)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        if not df.empty:
            raw_data[ticker] = df
            print(f'  ✅ {ticker:5s} ({name}): {len(df)} trading days')
        else:
            print(f'  ❌ {ticker:5s} ({name}): returned empty data')

    except Exception as e:
        print(f'  ❌ {ticker:5s} ({name}): {e}')

    time.sleep(1)

print(f'\n✅ Successfully downloaded {len(raw_data)}/{len(FINTECH_TICKERS)} stocks!')

Period: 2020-01-01 → 2024-12-31

  ✅ JPM   (JPMorgan Chase): 1257 trading days
  ✅ V     (Visa): 1257 trading days
  ✅ MA    (Mastercard): 1257 trading days
  ✅ PYPL  (PayPal): 1257 trading days
  ✅ XYZ   (Block (Square)): 1257 trading days
  ✅ COIN  (Coinbase): 935 trading days
  ✅ HOOD  (Robinhood): 861 trading days
  ✅ AFRM  (Affirm): 997 trading days

✅ Successfully downloaded 8/8 stocks!


## 2. Data Quality Check


In [23]:
print('='*55)
print('Ticker    Rows   Missing        Start          End')
print('='*55)
for ticker, df in raw_data.items():
    missing = df.isnull().sum().sum()
    start   = str(df.index[0].date())
    end     = str(df.index[-1].date())
    print(f'{ticker:<8} {len(df):>6} {missing:>8} {start:>12} {end:>12}')
print('='*55)

print('\nSample data (JPM):')
raw_data['JPM'].head()

Ticker    Rows   Missing        Start          End
JPM        1257        0   2020-01-02   2024-12-30
V          1257        0   2020-01-02   2024-12-30
MA         1257        0   2020-01-02   2024-12-30
PYPL       1257        0   2020-01-02   2024-12-30
XYZ        1257        0   2020-01-02   2024-12-30
COIN        935        0   2021-04-14   2024-12-30
HOOD        861        0   2021-07-29   2024-12-30
AFRM        997        0   2021-01-13   2024-12-30

Sample data (JPM):


Price,Close,High,Low,Open,Volume
Date,,,,,
2020-01-02,118.430328,118.438731,116.894233,117.339111,10803700
2020-01-03,116.867485,117.619343,115.803061,116.157869,10386800
2020-01-06,116.774536,116.808335,115.313063,115.363748,10259000
2020-01-07,114.789291,116.461961,114.738606,115.971984,10531300
2020-01-08,115.684784,116.225445,114.552776,114.637247,9695300


## 3. Feature Engineering


In [24]:
def add_technical_indicators(df):
    """Add comprehensive technical indicators to a stock DataFrame."""
    df = df.copy()
    close = df['Close']

    # --- Moving Averages ---
    df['SMA_20']  = close.rolling(window=20).mean()
    df['SMA_50']  = close.rolling(window=50).mean()
    df['SMA_200'] = close.rolling(window=200).mean()
    df['EMA_12']  = close.ewm(span=12, adjust=False).mean()
    df['EMA_26']  = close.ewm(span=26, adjust=False).mean()

    # --- MACD ---
    df['MACD']        = df['EMA_12'] - df['EMA_26']
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist']   = df['MACD'] - df['MACD_Signal']

    # --- RSI ---
    delta = close.diff()
    gain  = delta.clip(lower=0).rolling(window=14).mean()
    loss  = (-delta.clip(upper=0)).rolling(window=14).mean()
    rs    = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))

    # --- Bollinger Bands ---
    df['BB_Mid']   = close.rolling(window=20).mean()
    bb_std         = close.rolling(window=20).std()
    df['BB_Upper'] = df['BB_Mid'] + 2 * bb_std
    df['BB_Lower'] = df['BB_Mid'] - 2 * bb_std
    df['BB_Width'] = (df['BB_Upper'] - df['BB_Lower']) / df['BB_Mid']

    # --- Returns & Volatility ---
    df['Daily_Return']    = close.pct_change()
    df['Log_Return']      = np.log(close / close.shift(1))
    df['Volatility_20d']  = df['Daily_Return'].rolling(window=20).std() * np.sqrt(252)

    # --- Volume indicators ---
    df['Volume_MA20']  = df['Volume'].rolling(window=20).mean()
    df['Volume_Ratio'] = df['Volume'] / df['Volume_MA20']

    # --- Target variable ---
    df['Target'] = (close.shift(-1) > close).astype(int)  # 1 = price goes up tomorrow

    return df.dropna()

# Apply to all tickers
processed_data = {}
for ticker, df in raw_data.items():
    processed_data[ticker] = add_technical_indicators(df)
    print(f'✅ {ticker}: {len(processed_data[ticker])} rows after feature engineering')

print(f'\nFeatures added: {list(processed_data["JPM"].columns)}')

✅ JPM: 1058 rows after feature engineering
✅ V: 1058 rows after feature engineering
✅ MA: 1058 rows after feature engineering
✅ PYPL: 1058 rows after feature engineering
✅ XYZ: 1058 rows after feature engineering
✅ COIN: 736 rows after feature engineering
✅ HOOD: 662 rows after feature engineering
✅ AFRM: 798 rows after feature engineering

Features added: ['Close', 'High', 'Low', 'Open', 'Volume', 'SMA_20', 'SMA_50', 'SMA_200', 'EMA_12', 'EMA_26', 'MACD', 'MACD_Signal', 'MACD_Hist', 'RSI', 'BB_Mid', 'BB_Upper', 'BB_Lower', 'BB_Width', 'Daily_Return', 'Log_Return', 'Volatility_20d', 'Volume_MA20', 'Volume_Ratio', 'Target']


## 4. Visualize: Candlestick Chart with Indicators


In [25]:
def plot_candlestick(ticker, df, last_n=180):
    df_plot = df.tail(last_n)

    fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                        row_heights=[0.6, 0.2, 0.2],
                        subplot_titles=[f'{ticker} — Candlestick + Bollinger Bands',
                                        'Volume', 'RSI'])

    # Candlestick
    fig.add_trace(go.Candlestick(
        x=df_plot.index, open=df_plot['Open'], high=df_plot['High'],
        low=df_plot['Low'], close=df_plot['Close'], name='OHLC'), row=1, col=1)

    # Bollinger Bands
    for col, color, dash in [('BB_Upper','rgba(255,165,0,0.8)','dash'),
                               ('BB_Mid',  'rgba(255,165,0,0.5)','dot'),
                               ('BB_Lower','rgba(255,165,0,0.8)','dash')]:
        fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot[col],
                                  line=dict(color=color, dash=dash, width=1),
                                  name=col), row=1, col=1)

    # SMA lines
    fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['SMA_20'],
                              line=dict(color='cyan', width=1), name='SMA 20'), row=1, col=1)
    fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['SMA_50'],
                              line=dict(color='magenta', width=1), name='SMA 50'), row=1, col=1)

    # Volume
    colors = ['green' if r >= 0 else 'red' for r in df_plot['Daily_Return']]
    fig.add_trace(go.Bar(x=df_plot.index, y=df_plot['Volume'],
                          marker_color=colors, name='Volume'), row=2, col=1)

    # RSI
    fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['RSI'],
                              line=dict(color='yellow', width=1.5), name='RSI'), row=3, col=1)
    fig.add_hline(y=70, line_dash='dash', line_color='red',   row=3, col=1)
    fig.add_hline(y=30, line_dash='dash', line_color='green', row=3, col=1)

    fig.update_layout(template='plotly_dark', height=700,
                       title=f'{ticker} Technical Analysis (Last {last_n} Days)',
                       xaxis_rangeslider_visible=False)
    fig.show()

# Plot JPM as example
plot_candlestick('JPM', processed_data['JPM'])

## 5. Save Processed Data


In [26]:
for ticker, df in processed_data.items():
    path = f'../data/{ticker}_processed.csv'
    df.to_csv(path)
    print(f'💾 Saved: {path}')

# Also save a combined closing prices file
closes = pd.DataFrame({t: processed_data[t]['Close'] for t in processed_data})
closes.to_csv('../data/all_closes.csv')
print('💾 Saved: ../data/all_closes.csv')
print('\n✅ All data saved! Proceed to Notebook 02.')

💾 Saved: ../data/JPM_processed.csv
💾 Saved: ../data/V_processed.csv
💾 Saved: ../data/MA_processed.csv
💾 Saved: ../data/PYPL_processed.csv
💾 Saved: ../data/XYZ_processed.csv
💾 Saved: ../data/COIN_processed.csv
💾 Saved: ../data/HOOD_processed.csv
💾 Saved: ../data/AFRM_processed.csv
💾 Saved: ../data/all_closes.csv

✅ All data saved! Proceed to Notebook 02.
